Instalacja pakietu `pyjanitor`:

In [ ]:
%pip install pyjanitor

---

In [ ]:
# załadowanie bibliotek

import pandas as pd
import numpy as np
import janitor

In [ ]:
# wczytanie danych

df_pracownicy = pd.ExcelFile('data.xlsx').parse('pracownicy', index_col = 0)                    

In [ ]:
# zanieczyszczenie danych

import warnings

warnings.simplefilter(category = FutureWarning, action = 'ignore')

df_pracownicy_brudne = df_pracownicy.rename(columns = {'Stanowisko':'Zawód'}
        ).assign(**{'Imię'                 : lambda df: df['Imię'].where(df.index != 'AN01', 'Alex'),
                    'Płeć'                 : ['kobieta', 'mężczyzna', 'inna', 'kobieta', 'kobieta', 'mężczyzna', 'mężczyzna', 'kobieta', 'mężczyzna', 'mężczyzna', 'mężczyzna', 'mężczyzna'],
                    'Data urodzenia'       : lambda df: df['Data urodzenia'].astype('string'),
                    'Zarobki'              : lambda df: df.index.map(lambda x: 3100000   if x == 'BS99' else (
                                                                               '4400 zł' if x == 'LW89' else df.loc[x, 'Zarobki'])),
                    'Zawód'                : lambda df: df.index.map(lambda x: 'analityk  danych' if x == 'KM00' else (
                                                                               'Analityk'         if x == 'AN01' else (
                                                                               'analtyk'          if x == 'ŁG96' else (
                                                                               'analityk danych'  if x == 'AP98' else (
                                                                               'programista '     if x == 'AK94' else df.loc[x, 'Zawód']))))),
                    'Multisport'           : ['Tak', None, 'Nie', 'Nie', '-', 'Tak', 'Nie', 'Tak', 'Tak', 'Nie wiem', 'Nie', 'Tak'], 
                    'Hobby'                : ['Sport', 'Nauka', None, 'Inne: Sztuka', 'Sport', 'Podróże', 'Inne: Wędkarstwo', 'Inne: Historia', 'Nauka', 'Inne: kultura', None, 'Podróże'],
                    'Prawie pusta kolumna' : [None, None, 1490, None, 980, None, None, None, None, 3, None, None],
                    'Pusta kolumna'        : None,
                    'Jedna wartość'        : -7,
                    }
        ).pipe(lambda df: df.insert(df_pracownicy.columns.get_loc('Nazwisko') + 1, 'Płeć', df.pop('Płeć')) or df
        ).pipe(lambda df: pd.concat([df.loc['AK77'].to_frame().transpose(),
                          pd.concat([df.loc['LW89', :'Zawód'], pd.Series([pd.NA], index = ['Multisport']), df.loc['LW89', 'Hobby':]]).rename('LW89').to_frame().transpose(),
                          pd.concat([df.loc['AN01', :'Multisport'], pd.Series([None], index = ['Hobby']), df.loc['AN01', 'Prawie pusta kolumna':]]).rename('AN01').to_frame().transpose(),
                          df.loc['BS99'].to_frame().transpose(),
                          pd.concat([df.loc['AK94', :'Płeć'], pd.Series([pd.NA], index = ['Data urodzenia']), df.loc['AK94', 'Zarobki':]]).rename('AK94').to_frame().transpose(),
                          df.loc['JM00':'AP98'],
                          df.loc['JM00':'AP98'],
                          pd.concat([df.loc['WN88', :'Premia'], pd.Series([pd.NA], index = ['Zawód']), df.loc['WN88', 'Multisport':]]).rename('WN88').to_frame().transpose(),
                          df.loc['MS90':],
                          pd.DataFrame([np.hstack([[np.nan] * 10, None, np.nan])], index = ['AB12'], columns = df.columns)])
               )

df_pracownicy = df_pracownicy_brudne.copy()

---

<h1 style="color: red;">Czyszczenie danych</h1>

**<span style="color: darkblue;">Czyszczenie danych</span>** (ang. *data cleaning*) to przygotowanie danych do przetworzenia i analizy danych, ich uporządkowanie i uspójnienie

**<p style="color: darkblue;">Elementy procesu czyszczenia danych:</p>**

1. Uporządkowanie zbioru kolumn

- czyszczenie nazw
    - oczyszczenie stringów
    - zmiana niejasnych i niespójnych nazw
    - dodanie prefiksów do nazw pokrewnych kolumn
    
    
- zmiana kolejności kolumn

2. Usunięcie zbędnych wierszy i kolumn

- usunięcie pustych wierszy
- usunięcie pustych kolumn
- usunięcie kolumn z jedną wartością
- usunięcie wierszy duplikatów

3. Obsługa braków danych

4. Poprawa błędnych danych 

5. Poprawa typów kolumn



6.* Przekształcenie danych do postaci tidy


Najważniejszymi spośród nich są usunięcie błędnych danych, zmiana błędnych typów kolumn i usunięcie wierszy duplikatów, ponieważ często bez ich wykonania analiza nie jest możliwa.

**<span style="color: darkblue;">Uwaga!</span>** Nie wszystkie kroki można wykonać zawsze w tej samej kolejności, czasem trzeba wykonywać je naprzemiennie, powtarzając niektóre z nich.

**<h2 style="color: darkorange;">0. Wstępna inspekcja danych</h2>**

In [ ]:
# podgląd fragmentu danych



In [ ]:
# podstawowe informacje o danych 



**<h2 style="color: darkorange;">1. Uporządkowanie zbioru kolumn</h2>**

In [ ]:
# nazwy i kolejność kolumn



<h3 style="color: dodgerblue;">Czyszczenie nazw</h3>

Do procesu czyszczenia nazw należy zwykle m.in.:
- zamiana liter na małe
- zastąpienie spacji znakiem `_`
- usunięcie specyficznych znaków, jak polskie czy specjalne - pozostawienie samych liter, cyfr, `_` i `.`
- zmiana niejasnych lub niespójnych nazw na bardziej jasne lub spójne
- w przypadku kolumn należących do jakichś grup, dodanie do ich nazw prefiksu

**<span style="color: darkblue;">Oczyszczenie stringów</span>** - usunięcie wielkich litery, spacji, polskich znaków:

In [ ]:
# gdyby nie było pakietu janitor

df_pracownicy.rename(columns = dict(zip(df_pracownicy.columns,
                                        df_pracownicy.columns.str.strip(
                                                            ).str.replace('ł', 'l'
                                                            ).str.lower(
                                                            ).str.replace(' ', '_',
                                                            ).str.normalize('NFKD').str.encode('ascii', errors = 'ignore').str.decode('utf-8'))))

In [ ]:
# z pakietem janitor




**<p style="color: darkblue;">Zmiana niejasnych i niespójnych nazw:</p>**

**<p style="color: darkblue;">Dodanie prefiksów do nazw pokrewnych kolumn:</p>**

In [ ]:
cols_with_prefix = df_pracownicy.loc[:, 'imie':'data_urodzenia'].columns
prefix           = 'dane_osobiste.'

df_pracownicy.rename(columns = lambda x: prefix + x if x in cols_with_prefix else x,
                     #inplace = True
                     )

<h3 style="color: dodgerblue;">Zmiana kolejności kolumn</h3>

In [ ]:
# nowa kolejność:
# 'nazwisko', 'imie', 'plec', 'data_urodzenia', 'stanowisko', 'pensja', 'premia', 'czy_multisport',
# 'hobby', 'jedna_wartosc', 'prawie_pusta_kolumna', 'pusta_kolumna'




In [ ]:
# z grupą kolumn z prefiksem

#df_pracownicy =
df_pracownicy.rename(columns = lambda x: prefix + x if x in cols_with_prefix else x
            ).filter(np.hstack(['pensja',
                                prefix + cols_with_prefix,         # umieszczenie grupy kolumn
                                'stanowisko',
                                'premia']).tolist())

**<h2 style="color: darkorange;">2. Usunięcie zbędnych wierszy i kolumn</h2>**

Do zbędnych kolumn należą:
- puste kolumny
- kolumny z jedną wartością

In [ ]:
# liczba różnych wartości w kolumnach



Do zbędnych wierszy należą:
- puste wiersze
- wiersze duplikaty

In [ ]:
# liczba różnych wartości w wierszach



<h3 style="color: dodgerblue;">Usunięcie pustych wierszy i kolumn</h3>

In [ ]:
# gdy mamy domyślne indeksy



In [ ]:
# gdy mamy zdefiniowane indeksy;  NA - Not Available



<h3 style="color: dodgerblue;">Usunięcie kolumn z jedną wartością</h3>

In [2]:
def remove_constant(df):
    return df.loc[:, df.nunique() > 1]




<h3 style="color: dodgerblue;">Usunięcie wierszy duplikatów</h3>

In [ ]:
# czy wiersze są duplikatami



In [ ]:
# usunięcie duplikatów



Powyższe funkcje za duplikaty mają pary wierszy o tych samych wszystkich wartościach w kolumnach, niezależnie od tego, czy ich indeks się różni.

**<h2 style="color: darkorange;">3. Obsługa braków danych</h2>**

In [ ]:
# liczba braków danych w kolumnach



<h3 style="color: dodgerblue;">Ujednolicenie</h3>

pandas.dataframe może zawierać "pandasowy" brak danych - `pd.NA` (*Not Available*), "numpy-owy"- `np.nan` (*Not a Number*), lub "pythonowy" - `None`. Należy je ujednolicić, najlepiej do `np.nan`.

<h3 style="color: dodgerblue;">Eliminacja</h3>

Metody statystyczne zwykle nie uwzględniają możliwości braku danych. Jeśli więc czyszczenie wykonujemy na potrzeby uczenia maszynowego, gdzie w analizie bierze udział jednocześnie cały zbiór danych, powinniśmy je zwykle w pełni lub częściowo wyeliminować. 

W przypadku analizy opartej o testy statystyczne również zachodzi taka potrzeba, ale odnosi się ona jedynie do kolumn biorących udział w danym teście i zwykle obsługiwana jest automatycznie przez funkcję służącą do jego wykonania. W związku z tym, punkt ten zwykle można pominąć.

**<p style="color: darkblue;">Opcja 1: Usunięcie</p>**

- Usunięcie wierszy z co najmniej 1 brakiem

- Usunięcie kolumn z za dużą liczbą braków

In [ ]:
thresh_percent = 0.3                                               # co najmniej taki odsetek ma być niepustych wartości w kolumnie

df_pracownicy.dropna(thresh = thresh_percent * len(df_pracownicy), axis = 'columns', inplace = True)
df_pracownicy

Tak powstałe kolumny, zawierające wziąż w sobie braki, można wykorzystać, jeśli metoda uczenia maszynowego, z której zamierzamy skorzystać, posiada w sobie algorytm je obsługujący.

**<p style="color: darkblue;">Opcja 2: Imputacja (uzupełnienie)</p>**

In [ ]:
# z funkcją impute



In [ ]:
# z funkcją fillna



Metod imputacji brakujących danych wg określonych algorytmów jest wiele. Pozwala to na nieutracenie pozostałych wartości przy niedużej modyfikacji zebranych danych.

**<h2 style="color: darkorange;">4. Poprawa błędnych danych</h2>**

Aby wyłapać ze zbioru dane błędne i je wyeliminować, często trzeba przeanalizować wartości każdej z kolumn. W przypadku zmiennych kategorialnych pomocna jest przy tym tabela częstości (funkcja `value_counts`), a w przypadku zmiennych ilościowych główne statystyki opisowe (funkcja `describe`).

**<p style="color: darkblue;">Nazwisko</p>**

**<p style="color: darkblue;">Imię</p>**

**<p style="color: darkblue;">Płeć</p>**

**<p style="color: darkblue;">Data urodzenia</p>**

**<p style="color: darkblue;">Stanowisko</p>**

**<p style="color: darkblue;">Pensja</p>**

**<p style="color: darkblue;">Premia</p>**

**<p style="color: darkblue;">Czy multisport</p>**

**<p style="color: darkblue;">Hobby</p>**

In [ ]:
df_pracownicy['hobby'] = df_pracownicy['hobby'].str.lower(
                                              ).str.strip(
                                              ).replace({'inne: sztuka'     : 'kultura',
                                                         'inne: kultura'    : 'kultura',
                                                         'inne: wędkarstwo' : 'inne',
                                                         'inne: historia'   : 'inne'})
df_pracownicy['hobby'].value_counts(dropna = False) 

**<h2 style="color: darkorange;">5. Poprawa typów zmiennych</h2>**

Typy kolumn powinny być dopasowane do typu zmiennych jaki reprezentują, aby posiadać funkcjonalności, które posiadają zmienne na danej skali.

In [ ]:
# typy kolumn



<h3 style="color: dodgerblue;">Zmiana typu na <code>str</code>, <code>int</code>, <code>float</code>, <code>bool</code></h3>
- na zmienne nominalne, ilościowe, zero-jedynkowe

In [ ]:
# na str



In [ ]:
# na int



In [ ]:
# na float



In [ ]:
# na bool



*Uwaga*! Typ `bool` nie sprawdza się w praktyce jako typ badanej zmiennej - powinna być ona reprezentowana w danych jako zmienna zero-jedynkowa.

<h3 style="color: dodgerblue;">Zmiana typu na <code>category</code></h3>
- na zmienną porządkową

In [ ]:
# porządek: 'sprzedawca', 'analityk', 'programista', 'kierownik'




<h3 style="color: dodgerblue;">Zmiana typu na <code>datetime</code></h3>
- na zmienną czasową

`NaT` (*Not a Time*) - brak danych przeznaczony do typu `datatime`

<h3 style="color: dodgerblue;">Zmiana typów kilku zmiennych</h3>

In [ ]:
# zmiana typów

df_pracownicy.astype({'nazwisko'       : 'str',
                      'imie'           : 'str',
                      'plec'           : 'str',
                      'data_urodzenia' : 'datetime64[ns]',
                      'stanowisko'     : 'category',
                      'pensja'         : 'float',
                      'premia'         : 'float',
                      'czy_multisport' : 'bool',
                      'hobby'          : 'str'},
                     #inplace = True
                     )

In [ ]:
# sprawdzenie



In [ ]:
# zmiana typów z doprecyzowanymi typami category i datetime

df_pracownicy = df_pracownicy.astype({'nazwisko'       : 'str',
                                      'imie'           : 'str',
                                      'plec'           : 'str',
                                      'pensja'         : 'float',
                                      'premia'         : 'float',
                                      'czy_multisport' : 'float',
                                      'hobby'          : 'str'}
                            ).to_datetime('data_urodzenia', format = '%Y-%m-%d'
                            ).assign(stanowisko = df_pracownicy['stanowisko'].astype('category'
                                                                            ).cat.set_categories(['sprzedawca', 'analityk', 'programista', 'kierownik'],
                                                                                                 ordered = True))
df_pracownicy

In [ ]:
# obecne podstawowe informacje o danych



**<h2 style="color: darkorange;">6. Przekształcenie danych do postaci tidy</h2>**

W większości analiz, dane powinny być zapisane w tzw. postaci **<span style="color: darkblue;">tidy</span>**, w której:
 
1. jedna zmienna = jedna kolumna
2. jedna obserwacja = jeden wiersz
3. jedna wartość = jedna komórka
 
Jeśli dane są w innej postaci, możemy osiągnąć spełnianie powyższych warunków przez przekształcenie danych funkcjami takimi jak: \
`melt`, `pivot`, `transpose`, `stack`, `unstack`, `str.split`, `explode`, `str.cat`, `agg`. \
Zastosowanie pierwszych trzech z nich jest opisane w pliku 02.1_pandas_przetwarzanie.

In [ ]:
# podgląd fragmentu danych



---

**<h2 style="color: darkorange;">7. Podziwianie efektów</h2>**

Od tego zaczęliśmy:

In [ ]:
df_pracownicy_brudne

a na tym skończyliśmy:

In [ ]:
df_pracownicy

---

*<h3 style="color: forestgreen;"> Ćwiczenie ✏️ </h3>*

Wyczyść dane znajdujące się w pliku `data_messed.xlsx` w arkuszu `messy_data`: